In [5]:
#%load_ext cudf.pandas
#%load_ext cuml.accel

import cudf
import pandas as pd
import cupy as cp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tqdm
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [6]:
EMBED_DIM = 128

In [29]:
q1_emb = np.loadtxt('datasets/Quora Questions Pair Dataset/08_q1_emb.csv', delimiter=",")
q2_emb = np.loadtxt('datasets/Quora Questions Pair Dataset/09_q2_emb.csv', delimiter=",")
y = pd.read_csv('datasets/Quora Questions Pair Dataset/03_y.csv')

In [8]:
q1_emb.shape, q2_emb.shape, y.shape

((403968, 128), (403968, 128), (403968, 1))

In [9]:
q1_emb.shape, q1_emb

((403968, 128),
 array([[ 0.399972, -0.03023 ,  0.478363, ...,  0.051892,  0.057697,
          0.057988],
        [ 0.229123,  0.168725,  0.507309, ...,  0.026152, -0.179332,
         -0.202087],
        [-0.010637, -0.01014 ,  0.531773, ...,  0.006311,  0.084791,
         -0.000958],
        ...,
        [ 0.086741,  0.055834,  0.491166, ...,  0.043546,  0.023323,
         -0.150595],
        [ 0.224093,  0.01481 ,  0.568347, ..., -0.093007,  0.11021 ,
         -0.040135],
        [ 0.100657,  0.024295,  0.693396, ..., -0.140602,  0.084881,
         -0.130592]], shape=(403968, 128)))

In [10]:
q1_emb[0, :].shape, q1_emb[0, :]

((128,),
 array([ 3.99972e-01, -3.02300e-02,  4.78363e-01,  8.42790e-02,
        -1.81888e-01, -4.08740e-02,  3.75800e-03,  4.50580e-02,
        -1.21358e-01,  8.45500e-03,  1.68237e-01, -5.77030e-02,
         8.30000e-05, -2.03857e-01,  1.97170e-02,  1.19703e-01,
         1.79360e-01, -6.00750e-02,  7.48110e-02,  1.46214e-01,
        -2.83320e-02,  7.40000e-04, -1.15288e-01, -3.24720e-02,
         1.06683e-01, -6.12030e-02, -2.47930e-02, -2.99900e-03,
         1.91896e-01,  1.95580e-01, -8.80500e-03,  1.99817e-01,
         2.52810e-02, -6.27220e-02, -7.57720e-02,  8.14690e-02,
         2.45910e-02,  1.01934e-01, -1.70095e-01, -1.46690e-01,
        -1.57160e-01, -1.64433e-01,  8.83210e-02,  4.62320e-02,
        -7.54840e-02,  2.26090e-02,  3.60790e-02, -2.59761e-01,
        -1.55183e-01, -1.05439e-01,  7.28000e-04,  4.17230e-02,
        -1.67337e-01, -5.22160e-02, -7.63140e-02,  4.14241e-01,
        -4.78700e-02,  5.52240e-02, -7.10780e-02, -2.68034e-01,
         1.01091e-01,  7.25150e

In [11]:
q1_emb[0, :].reshape(1,-1).shape, q1_emb[0, :].reshape(1,-1)

((1, 128),
 array([[ 3.99972e-01, -3.02300e-02,  4.78363e-01,  8.42790e-02,
         -1.81888e-01, -4.08740e-02,  3.75800e-03,  4.50580e-02,
         -1.21358e-01,  8.45500e-03,  1.68237e-01, -5.77030e-02,
          8.30000e-05, -2.03857e-01,  1.97170e-02,  1.19703e-01,
          1.79360e-01, -6.00750e-02,  7.48110e-02,  1.46214e-01,
         -2.83320e-02,  7.40000e-04, -1.15288e-01, -3.24720e-02,
          1.06683e-01, -6.12030e-02, -2.47930e-02, -2.99900e-03,
          1.91896e-01,  1.95580e-01, -8.80500e-03,  1.99817e-01,
          2.52810e-02, -6.27220e-02, -7.57720e-02,  8.14690e-02,
          2.45910e-02,  1.01934e-01, -1.70095e-01, -1.46690e-01,
         -1.57160e-01, -1.64433e-01,  8.83210e-02,  4.62320e-02,
         -7.54840e-02,  2.26090e-02,  3.60790e-02, -2.59761e-01,
         -1.55183e-01, -1.05439e-01,  7.28000e-04,  4.17230e-02,
         -1.67337e-01, -5.22160e-02, -7.63140e-02,  4.14241e-01,
         -4.78700e-02,  5.52240e-02, -7.10780e-02, -2.68034e-01,
          1.01

## Feature Engineering

In [12]:
# Experimental
a = q1_emb[0, :].reshape(1,-1)
b = q2_emb[0, :].reshape(1,-1)

In [13]:
# Experimental
# cosine similarity - method 1
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity(a, b)

array([[0.9852595]])

In [14]:
# Experimental
# cosine similarity - method 2
a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
(a_n * b_n).sum(axis=1)

array([0.98525949])

In [15]:
# Experimental
# cosine similarity - method 3
from scipy.spatial import distance
1 - distance.cosine(a.reshape(128),b.reshape(128))

np.float64(0.985259504299558)

In [16]:
def cosine_sim(a,b):
    return cosine_similarity(a.reshape(1,-1), b.reshape(1,-1))[0][0]

In [17]:
cos_sim = []
for i in tqdm(range(q1_emb.shape[0]), desc='calculating cosine similarity...'):
    q1 = q1_emb[i]
    q2 = q2_emb[i]
    cos_sim.append(cosine_sim(q1, q2))

calculating cosine similarity...: 100%|█████████████████████████████████████| 403968/403968 [02:49<00:00, 2385.22it/s]


In [18]:
cos_sim = np.array(cos_sim)
cos_sim.shape, cos_sim

((403968,),
 array([0.9852595 , 0.89129902, 0.73099077, ..., 0.9029046 , 0.30859193,
        0.96252111], shape=(403968,)))

In [19]:
euclid_dist = np.linalg.norm(q1_emb - q2_emb, axis=1)
euclid_dist.shape, euclid_dist

((403968,),
 array([0.2775875 , 1.10931517, 1.12516229, ..., 0.81415368, 1.90839657,
        0.40302498], shape=(403968,)))

In [20]:
df_sim = pd.DataFrame({
    'cosine_similarity': cos_sim,
    'euclid_dist': euclid_dist
})
df_sim.shape, df_sim

((403968, 2),
         cosine_similarity  euclid_dist
 0                0.985260     0.277588
 1                0.891299     1.109315
 2                0.730991     1.125162
 3                0.283524     2.574903
 4                0.631059     2.023907
 ...                   ...          ...
 403963           0.882167     0.942720
 403964           0.842347     0.859353
 403965           0.902905     0.814154
 403966           0.308592     1.908397
 403967           0.962521     0.403025
 
 [403968 rows x 2 columns])

In [21]:
df_emb = pd.DataFrame(np.concat([q1_emb, q2_emb], axis=1), columns=[f'e_{i}' for i in range(EMBED_DIM * 2)])
df_emb.shape, df_emb

((403968, 256),
              e_0       e_1       e_2       e_3       e_4       e_5       e_6  \
 0       0.399972 -0.030230  0.478363  0.084279 -0.181888 -0.040874  0.003758   
 1       0.229123  0.168725  0.507309  0.033915 -0.493944 -0.054575  0.017291   
 2      -0.010637 -0.010140  0.531773 -0.181084  0.137311  0.130822 -0.021594   
 3      -0.096622  0.049646  0.354710 -0.111045 -0.180998 -0.069335 -0.237087   
 4       0.042259 -0.005494  0.495534 -0.109559 -0.114276  0.058373  0.290820   
 ...          ...       ...       ...       ...       ...       ...       ...   
 403963  0.054989  0.070236  0.667844  0.116850 -0.168371  0.063109  0.213068   
 403964  0.164337  0.106398  0.457740  0.026822  0.038998 -0.041263  0.049956   
 403965  0.086741  0.055834  0.491166 -0.171129 -0.173165  0.017924  0.176467   
 403966  0.224093  0.014810  0.568347  0.202049 -0.149187 -0.175398  0.005155   
 403967  0.100657  0.024295  0.693396  0.202325 -0.038961 -0.051401 -0.045135   
 
          

In [22]:
df_diff = pd.DataFrame(np.abs(q1_emb - q2_emb), columns=[f'diff_{i}' for i in range(EMBED_DIM)])
df_diff.shape, df_diff

((403968, 128),
           diff_0    diff_1    diff_2    diff_3    diff_4    diff_5    diff_6  \
 0       0.023052  0.037022  0.034096  0.016243  0.010736  0.015283  0.050454   
 1       0.016248  0.061548  0.054150  0.010255  0.072860  0.107044  0.152752   
 2       0.183528  0.118940  0.195970  0.021483  0.093977  0.006792  0.049693   
 3       0.252022  0.013025  0.074218  0.182067  0.029752  0.284164  0.099395   
 4       0.237803  0.052568  0.140658  0.066839  0.064373  0.105430  0.051311   
 ...          ...       ...       ...       ...       ...       ...       ...   
 403963  0.036338  0.088997  0.031267  0.004163  0.138584  0.152207  0.117117   
 403964  0.102401  0.011618  0.043953  0.044302  0.055884  0.037002  0.051484   
 403965  0.014474  0.004438  0.049340  0.103032  0.057286  0.001529  0.067914   
 403966  0.258405  0.121220  0.125812  0.282256  0.065394  0.084918  0.051084   
 403967  0.047669  0.048158  0.041177  0.016664  0.005488  0.025798  0.007282   
 
          

In [30]:
df = pd.concat([df_sim, df_diff, df_emb, y], axis=1)

In [32]:
df.to_csv('datasets/Quora Questions Pair Dataset/10_Xy.csv', index=False)